In [ ]:
from binary_ext_fields.custom_field import TableField, create_field
from binary_ext_fields.generate_symbols import generate_symbols_random, check_orth, recode, check_orth_skip_coeffs
from binary_ext_fields.orthogonal_tag_creator import OrthogonalTagGenerator as OTC

from utils.log_helpers import print_generation, to_int_matrx

from icecream import ic

## generate some random Symbols in our chosen field and use the identy matrix as coefficents

In [ ]:
accepts = False
while not accepts:

    data_fields = 3
    gen_size = 5
    field_int = 3

    table_field = create_field(field_int)
    tag_gen = OTC(table_field)

    generation = generate_symbols_random(0,table_field.max_value,data_fields,gen_size)
    ic(generation)
    # new matrix with Identity matrix in the front

    gen_w_coefficients = []
    generation_size = len(generation)

    coefficients = bytearray(generation_size)


    for i, pkt in enumerate(generation):
        coefficients = bytearray(generation_size)
        coefficients[i] = 1

        gen_w_coefficients.append(coefficients.copy() + pkt.copy())


    tagged_gen = tag_gen.generate_all_tags(gen_w_coefficients)

    accepts = (check_orth(table_field, tagged_gen)) # loop until a generation doesnt have the 0 tag error

    # test recode function
    recoded_packets = []
    for i in range(5):
        recoded_packets.append(recode(table_field, tagged_gen, 1))
    ic(recoded_packets)

print_generation(recoded_packets)
print_generation(tagged_gen)

ic(check_orth(table_field, recoded_packets))



long_matrix = []
nr_recodes = 10


for i in range(nr_recodes):
    new_packet = recode(table_field, tagged_gen, 1)
    long_matrix.append(new_packet)

ic(long_matrix)
long_matrix_int = to_int_matrx(long_matrix)
print_generation(long_matrix_int)




In [ ]:
# now lets do real rlnc recoding
from binary_ext_fields.generate_symbols import recode_rlnc
ic(generation)

data_fields = 3
gen_size = 5
field_int = 3
table_field = create_field(field_int)
tag_gen = OTC(table_field)

tagged_gen = []
accepts = False

while not accepts:

    generation = generate_symbols_random(0,table_field.max_value,data_fields,gen_size)
    tagged_gen =  tag_gen.generate_all_tags(generation)
    accepts = (check_orth(table_field, tagged_gen)) # loop until a generation doesnt have the 0 tag error

rlnc_recoded_packets = recode_rlnc(table_field, tagged_gen, gen_size, 15)


In [ ]:
# check orthogonality of recoded packets:
from pathlib import Path
ic(rlnc_recoded_packets)
bool1 = check_orth(table_field,rlnc_recoded_packets,Path("D:\projects\studienarbeit\logs\oerror_no_coeff.txt"))
bool2 = check_orth_skip_coeffs(table_field,rlnc_recoded_packets,gen_size, Path("D:\projects\studienarbeit\logs\oerror_coeff.txt"))
ic(rlnc_recoded_packets)
ic(bool1, bool2)


In [ ]:
# make the ARC check with rref implementation
from binary_ext_fields.rref import calculate_rref

rref = calculate_rref(rlnc_recoded_packets,table_field,gen_size)

In [ ]:
# now well look how this works out with an error in the matrix
import sys
#first with a packet later in the generation
# make copy of original matrix first

rlnc_recoded_packets_copy = rlnc_recoded_packets.copy()

error_packet = rlnc_recoded_packets_copy[11].copy()
ic(error_packet, rlnc_recoded_packets_copy[11])
error_packet[2] = ic(error_packet[2] + 1)
ic(error_packet, rlnc_recoded_packets_copy[11])

rlnc_recoded_packets_copy[11] = error_packet

ic(rlnc_recoded_packets_copy)

rref_with_error = calculate_rref(rlnc_recoded_packets_copy, table_field, gen_size)
ic(rref_with_error)

In [ ]:
# check orth with error packet in generation

bool1 = check_orth(table_field,rlnc_recoded_packets_copy,Path("D:\projects\studienarbeit\logs\error_logthis.txt"))
bool2 = check_orth_skip_coeffs(table_field,rlnc_recoded_packets_copy,gen_size, Path("D:\projects\studienarbeit\logs\error_logthat.txt"))
ic(bool1, bool2)

In [ ]:
# now lets try en error in an earlier packet
ic(rlnc_recoded_packets)
rlnc_recoded_packets_copy_2 = rlnc_recoded_packets.copy()

error_packet = rlnc_recoded_packets_copy_2[3].copy()
ic(error_packet, rlnc_recoded_packets_copy_2[3])
error_packet[2] = ic(error_packet[2] + 1)
ic(error_packet, rlnc_recoded_packets_copy_2[3])

rlnc_recoded_packets_copy_2[3] = error_packet

ic(rlnc_recoded_packets_copy_2)
orthogonal_bool_1 = check_orth(table_field, rlnc_recoded_packets_copy_2,Path("D:\projects\studienarbeit\logs\early_error.txt"))
orthogonal_bool_2 = check_orth_skip_coeffs(table_field, rlnc_recoded_packets_copy_2,gen_size, Path("D:\projects\studienarbeit\logs\early_error_skipped_coeff.txt"))

rref_with_error = calculate_rref(rlnc_recoded_packets_copy_2, table_field, gen_size)
ic(rref_with_error, orthogonal_bool_1, orthogonal_bool_2)


In [ ]:
# for one last test check the tagging after adding coeffs and check all orths again
from binary_ext_fields.generate_symbols import recode_rlnc_without_coeffs
data_fields = 3
gen_size = 5
field_int = 3
table_field = create_field(field_int)
tag_gen = OTC(table_field)

tagged_gen = []
accepts = False

while not accepts:

    generation = generate_symbols_random(0,table_field.max_value,data_fields,gen_size)
    tagged_gen =  tag_gen.generate_all_tags(generation)
    accepts = (check_orth(table_field, tagged_gen)) # loop until a generation doesnt have the 0 tag error

rlnc_recoded_packets = recode_rlnc_without_coeffs(table_field, tagged_gen, gen_size, 15)

In [ ]:
ic(rlnc_recoded_packets)
check_orth(table_field, rlnc_recoded_packets, Path("D:\projects\studienarbeit\logs\coeffmixup.txt"))
check_orth_skip_coeffs(table_field, rlnc_recoded_packets, gen_size, Path("D:\projects\studienarbeit\logs\coeffmixup_withcoeff.txt"))

In [ ]:
from utils.log_helpers import log_generation_detail

log_generation_detail(rlnc_recoded_packets,table_field,Path("D:\projects\studienarbeit\logs\generation_detail.txt") )
log_generation_detail(rlnc_recoded_packets,table_field)
